# Test Agentics Search API

This notebook tests the search functionality to understand how Agentics `.search()` method works.

In [ ]:
from agentics import AG
import pandas as pd
from pathlib import Path

## 1. Load a CSV file and create AG corpus

In [ ]:
# Find a CSV file in the data directory
csv_path = "../data/ibm_2025_report.csv"

# Check if file exists
if Path(csv_path).exists():
    print(f"✓ Found CSV file: {csv_path}")
else:
    print(f"✗ CSV file not found: {csv_path}")
    print("Available files in data/:")
    for f in Path("data").glob("*.csv"):
        print(f"  - {f}")

In [ ]:
# Load the CSV to see its structure
df = pd.read_csv(csv_path)
print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
df.head(3)

In [ ]:
# Create AG corpus from CSV
print("Creating AG corpus...")
corpus = AG.from_csv(csv_path, max_rows=100)
print(f"✓ Corpus created")
print(f"Corpus type: {type(corpus)}")
print(f"Corpus has df: {hasattr(corpus, 'df')}")
if hasattr(corpus, 'df'):
    print(f"Corpus df shape: {corpus.df.shape}")

## 2. Test the search method

In [ ]:
# Test search with different entities
test_entities = ["accuracy", "IBM", "revenue", "technology"]

for entity in test_entities:
    print(f"\n{'='*60}")
    print(f"Searching for: '{entity}'")
    print(f"{'='*60}")
    
    # Perform search
    result = corpus.search(entity, k=5)
    
    print(f"\nResult type: {type(result)}")
    print(f"Result attributes: {[attr for attr in dir(result) if not attr.startswith('_')]}")
    print(f"Has df: {hasattr(result, 'df')}")
    
    if hasattr(result, 'df'):
        df_result = result.df
        print(f"df type: {type(df_result)}")
        print(f"df is None: {df_result is None}")
        
        if df_result is not None:
            print(f"df shape: {df_result.shape}")
            print(f"df columns: {list(df_result.columns)}")
            print(f"\nFirst result:")
            if len(df_result) > 0:
                print(df_result.iloc[0])
            else:
                print("No results found")
    
    # Try to access as list
    print(f"\nIs iterable: {hasattr(result, '__iter__')}")
    print(f"Has __len__: {hasattr(result, '__len__')}")
    if hasattr(result, '__len__'):
        print(f"Length: {len(result)}")

## 3. Test with a specific entity and examine the result structure

In [ ]:
# Deep dive into one search result
entity = "accuracy"
print(f"Deep analysis of search for '{entity}':\n")

result = corpus.search(entity, k=10)

# Check all possible ways to access data
print("1. Direct attributes:")
for attr in ['df', 'data', 'results', 'passages', 'documents']:
    if hasattr(result, attr):
        val = getattr(result, attr)
        print(f"   {attr}: {type(val)} - {val is not None}")
        if val is not None and hasattr(val, 'shape'):
            print(f"      shape: {val.shape}")

print("\n2. Try iteration:")
try:
    for i, item in enumerate(result):
        print(f"   Item {i}: {type(item)}")
        if i >= 2:  # Just show first 3
            break
except Exception as e:
    print(f"   Cannot iterate: {e}")

print("\n3. Try indexing:")
try:
    first = result[0]
    print(f"   result[0]: {type(first)}")
    print(f"   {first}")
except Exception as e:
    print(f"   Cannot index: {e}")

print("\n4. String representation:")
print(f"   {str(result)[:500]}...")  # First 500 chars

## 4. Compare with direct dataframe search

In [ ]:
# Try searching in the original dataframe
entity = "accuracy"
print(f"Searching for '{entity}' in original dataframe:\n")

# Search in all text columns
text_cols = df.select_dtypes(include=['object']).columns
print(f"Text columns: {list(text_cols)}\n")

for col in text_cols:
    matches = df[df[col].str.contains(entity, case=False, na=False)]
    if len(matches) > 0:
        print(f"Found {len(matches)} matches in column '{col}'")
        print(f"First match: {matches.iloc[0][col][:200]}...\n")

## 5. Test the actual API endpoint

In [ ]:
import requests
import json

API_BASE = "http://localhost:5000/api"

# First, check if server is running
try:
    response = requests.get(f"{API_BASE}/status")
    print(f"Server status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
except Exception as e:
    print(f"Server not running or error: {e}")
    print("\nMake sure to start the server with: python agentics_kg/app.py")

In [ ]:
# Test search endpoint
search_data = {
    "entity": "accuracy",
    "top_n": 5
}

try:
    response = requests.post(f"{API_BASE}/search", json=search_data)
    print(f"Search status: {response.status_code}")
    result = response.json()
    print(f"\nResponse:")
    print(json.dumps(result, indent=2))
    
    if result.get('success'):
        print(f"\nFound {result['count']} passages")
        if result['count'] > 0:
            print(f"\nFirst passage:")
            print(json.dumps(result['passages'][0], indent=2))
except Exception as e:
    print(f"Error: {e}")

## Summary

This notebook helps identify:
1. How AG.search() returns data
2. What attributes are available on the result
3. How to properly extract passages
4. Whether the API endpoint is working correctly